# 08 — Generation Dispatch Mix & Emissions

**Purpose:** Visualise the fuel-level generation dispatch and CO₂ emissions
for each completed E4ST scenario.  All quantities are read directly from
Julia output — nothing is recomputed here.

The copper-plate model dispatches 11 fuel types across 67 BA zones.
Investment capacity additions are zero for all completed scenarios
(the investment module was not activated in this run).

**Inputs:**
- `data/processed/e4st_results/{scenario}/dispatch.parquet`
- `data/processed/network_metadata.json` — scenario registry

**Outputs:**
- `data/processed/dispatch_mix.png` — aggregate generation mix by scenario
- `data/processed/dispatch_co2.png` — CO₂ emissions by fuel and scenario
- `data/processed/dispatch_ba_heatmap.png` — per-BA fuel share heatmap

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
RESULTS_DIR = PROJECT_ROOT / 'data' / 'processed' / 'e4st_results'
META_PATH   = PROJECT_ROOT / 'data' / 'processed' / 'network_metadata.json'

with open(META_PATH) as f:
    meta = json.load(f)

SCENARIOS = [
    s['name']
    for s in meta['scenarios_completed']
    if s['status'] == 'OPTIMAL'
]

SCENARIO_LABELS = {
    'baseline':       'Baseline',
    'carbon_tax_50':  'Carbon Tax\n$50/tCO₂',
    'ces_achievable': 'CES 50%\n(max feasible)',
}

# ── Fuel colour scheme — consistent with 01_eia_pull FUEL_COLORS ──────────────
# Keys match the `genfuel` column values in dispatch.parquet
FUEL_COLORS = {
    'coal':       '#2d2d2d',   # near-black
    'ng':         '#4169e1',   # royal blue  ("natural gas")
    'nuclear':    '#d62728',   # red
    'wind':       '#2ca02c',   # green
    'solar':      '#ff7f0e',   # orange
    'hydro':      '#17becf',   # cyan
    'biomass':    '#8c564b',   # brown
    'oil':        '#9467bd',   # purple
    'geothermal': '#e377c2',   # pink
    'storage':    '#bcbd22',   # yellow-green
    'other':      '#7f7f7f',   # gray
}

FUEL_ORDER = [
    'ng', 'coal', 'nuclear', 'hydro', 'wind', 'solar',
    'biomass', 'geothermal', 'oil', 'storage', 'other'
]

print('Scenarios:', SCENARIOS)

In [ ]:
# ── Load dispatch for all scenarios ───────────────────────────────────────────
frames = []
for sc in SCENARIOS:
    df = pd.read_parquet(RESULTS_DIR / sc / 'dispatch.parquet')
    df['scenario'] = sc
    frames.append(df)
dispatch = pd.concat(frames, ignore_index=True)

# Convert MWh → TWh for readability
dispatch['dispatch_twh'] = dispatch['dispatch_mwh'] / 1e9
dispatch['co2_mtons']    = dispatch['co2_emitted_tons'] / 1e9  # metric megatons CO2

print('Loaded rows:', len(dispatch))
print('Fuel types: ', sorted(dispatch['genfuel'].unique()))

In [ ]:
# ── Aggregate: generation by scenario × fuel ───────────────────────────────────
agg = (
    dispatch
    .groupby(['scenario', 'genfuel'])[['dispatch_twh', 'co2_mtons']]
    .sum()
    .reset_index()
)

# Pivot for stacked bars
fuels_present = [f for f in FUEL_ORDER if f in agg['genfuel'].unique()]
gen_pivot = (
    agg.pivot(index='scenario', columns='genfuel', values='dispatch_twh')
    .reindex(columns=fuels_present)
    .fillna(0)
)
co2_pivot = (
    agg.pivot(index='scenario', columns='genfuel', values='co2_mtons')
    .reindex(columns=fuels_present)
    .fillna(0)
)

print('Generation pivot (TWh):')
gen_pivot.round(3)

In [ ]:
# ── Figure 1: Aggregate generation mix by scenario ────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(SCENARIOS))
bottoms = np.zeros(len(SCENARIOS))

for fuel in fuels_present:
    vals = gen_pivot.reindex(SCENARIOS)[fuel].values
    ax.bar(
        x, vals, bottom=bottoms,
        color=FUEL_COLORS.get(fuel, '#aaaaaa'),
        label=fuel, edgecolor='white', linewidth=0.4,
    )
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels([SCENARIO_LABELS.get(s, s) for s in SCENARIOS])
ax.set_ylabel('Annual Generation (TWh)')
ax.set_title('E4ST Zonal Model — Generation Mix by Scenario')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}'))

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          bbox_to_anchor=(1.01, 1), loc='upper left',
          frameon=False, fontsize=9)

fig.tight_layout()
out = PROJECT_ROOT / 'data' / 'processed' / 'dispatch_mix.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved → {out}')
plt.show()

In [ ]:
# ── Figure 2: CO₂ emissions by fuel and scenario ──────────────────────────────
# Only fuels that actually emit CO2
co2_fuels = [f for f in fuels_present if co2_pivot[f].sum() > 0]

fig, ax = plt.subplots(figsize=(8, 5))

bottoms = np.zeros(len(SCENARIOS))
for fuel in co2_fuels:
    vals = co2_pivot.reindex(SCENARIOS)[fuel].values
    ax.bar(
        x, vals, bottom=bottoms,
        color=FUEL_COLORS.get(fuel, '#aaaaaa'),
        label=fuel, edgecolor='white', linewidth=0.4,
    )
    # Annotate each segment if large enough
    for xi, (bot, val) in enumerate(zip(bottoms, vals)):
        if val > 0.01:
            ax.text(xi, bot + val / 2, f'{val*1000:.0f}',
                    ha='center', va='center', fontsize=7.5,
                    color='white' if fuel in ('coal', 'ng') else 'black')
    bottoms += vals

ax.set_xticks(x)
ax.set_xticklabels([SCENARIO_LABELS.get(s, s) for s in SCENARIOS])
ax.set_ylabel('CO₂ Emissions (Mt CO₂)')
ax.set_title('E4ST Zonal Model — Annual CO₂ Emissions by Scenario')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v*1000:.0f} Mt'))

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[::-1], labels[::-1],
          bbox_to_anchor=(1.01, 1), loc='upper left',
          frameon=False, fontsize=9)

fig.tight_layout()
out = PROJECT_ROOT / 'data' / 'processed' / 'dispatch_co2.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved → {out}')
plt.show()

In [ ]:
# ── Figure 3: Per-BA fuel share heatmap (baseline) ────────────────────────────
# Fuel share = % of each BA's total generation from each fuel
base = dispatch[dispatch['scenario'] == 'baseline'].copy()

ba_gen = (
    base
    .groupby(['ba', 'genfuel'])['dispatch_mwh']
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=fuels_present, fill_value=0)
)

# Normalise rows to 100 %
ba_share = ba_gen.div(ba_gen.sum(axis=1), axis=0) * 100

# Sort BAs by total coal+ng share ("most thermal" first)
ba_share['_thermal'] = ba_share.get('coal', 0) + ba_share.get('ng', 0)
ba_share = ba_share.sort_values('_thermal', ascending=False).drop(columns='_thermal')

fig, ax = plt.subplots(figsize=(10, 14))
im = ax.imshow(ba_share.values, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=100)

ax.set_xticks(range(len(fuels_present)))
ax.set_xticklabels(fuels_present, rotation=40, ha='right', fontsize=9)
ax.set_yticks(range(len(ba_share)))
ax.set_yticklabels(ba_share.index, fontsize=7.5)
ax.set_title('Baseline Generation Mix per BA (% of BA total)', pad=10)

plt.colorbar(im, ax=ax, shrink=0.5, label='Share of BA generation (%)')
fig.tight_layout()

out = PROJECT_ROOT / 'data' / 'processed' / 'dispatch_ba_heatmap.png'
fig.savefig(out, bbox_inches='tight')
print(f'Saved → {out}')
plt.show()

In [ ]:
# ── Dispatch summary table ────────────────────────────────────────────────────
total_gen = agg.groupby('scenario')['dispatch_twh'].sum().rename('Total Gen (TWh)')
total_co2 = (
    agg.groupby('scenario')['co2_mtons'].sum()
    .mul(1000).rename('Total CO₂ (Mt)')
)
ci = (total_co2 * 1e6 / (total_gen * 1e9)).rename('Carbon Intensity (tCO₂/MWh)')

summary = pd.concat([total_gen, total_co2, ci], axis=1).reindex(SCENARIOS)
summary.index = [SCENARIO_LABELS.get(s, s).replace('\n', ' ') for s in summary.index]
summary.style.format({'Total Gen (TWh)': '{:.1f}',
                      'Total CO₂ (Mt)':  '{:.0f}',
                      'Carbon Intensity (tCO₂/MWh)': '{:.4f}'})